Gradient Boosting Classifier

Part 1: Predicting if a Storm Would Occur using Gradient Boosting Classifier

Model:
    Objective: 
        min Z1 classification error, when predicting tropical storms
    Constraints:
        CO2 emisions
        Monthly temps (Jan to Dec)
        Year
        Month
        Day

        Month should between Jan-Dec
            If month Jan, Mar, May, Jul, Aug, Oct, Dec
	            day>= 1 && day<=31
            If month Apr, Jun, Sep, Nov
	            day>= 1 && day<=30
            If month Feb
	            day>=1 && day <=28
            If year %4 && year %100 &year %400
		        day>= 1 && day<=29	

Expected output - binary value 1 for tropical storm occurred else 0

In [79]:
#all imports
import pandas as pd
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
import time
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error, r2_score, classification_report
from sklearn.preprocessing import LabelEncoder
#from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt

In [80]:
#Reading from the dataset
#data = pd.read_csv("completed_dataset_for_IS_project_25_v2.csv")
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

#testing if the reading from the dataset was successful
print(data.head(5))

   Year  MONTH  DAY   LAT  LONG  WIND_KTS  PRESSURE CAT  Shape_Leng Country  \
0  1880      8   11  23.0 -91.9        70         0  H1    0.806226  Mexico   
1  1880      8   11  23.4 -92.6        80         0  H1    0.761577  Mexico   
2  1880      8   11  23.7 -93.3        80         0  H1    0.583095  Mexico   
3  1880      8   12  24.0 -93.8        90         0  H2    0.670820  Mexico   
4  1880      9    6  23.9 -88.6        40         0  TS    0.360555  Mexico   

   ...   Jun   Jul   Aug   Sep   Oct   Nov   Dec  Storm Intensity  \
0  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         56.43582   
1  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.92616   
2  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         46.64760   
3  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.37380   
4  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         14.42220   

   Storm Intensity Label  Wind Speed Squared  
0                      3                4900  
1               

In [81]:
'''
def valid_date(row):
    month = row ["MONTH"]
    day = row ["DAY"]
    year = row["Year"]

    if month < 1 or month > 12:
        return False
    if month in [1,3,5,7,8,10,12] and not (1<=day <= 31):
        return False
    if month in [4,6,9,11] and not (1<= day <=30):
        return False
    if month == 2:
        leap_year = (year %4 == 0 and year % 100 != 0) or (year %400 == 0)
        possible_day = 29 if leap_year else 28

        if not (1 <= day <= possible_day):
            return False
    return True
    
    #removing in valid dates from the dataset 
    #data = data[data.apply(valid_date, axis=1)]
'''

'\ndef valid_date(row):\n    month = row ["MONTH"]\n    day = row ["DAY"]\n    year = row["Year"]\n\n    if month < 1 or month > 12:\n        return False\n    if month in [1,3,5,7,8,10,12] and not (1<=day <= 31):\n        return False\n    if month in [4,6,9,11] and not (1<= day <=30):\n        return False\n    if month == 2:\n        leap_year = (year %4 == 0 and year % 100 != 0) or (year %400 == 0)\n        possible_day = 29 if leap_year else 28\n\n        if not (1 <= day <= possible_day):\n            return False\n    return True\n    \n    #removing in valid dates from the dataset \n    #data = data[data.apply(valid_date, axis=1)]\n'

Since our dataset only has data when tropical storms occured inorder to use the classifier to train the models, we'll have to use "dummy values" in order to train the model for the classifier to learn te difference. 

Generating realistic "dummy values" by  using the dates what storms did not occur and initilizing information about the storms to be 0 based on their data types

In [82]:
'''
data["storm_occured"] = 1

#possible dates
years = range(data["Year"].min(), data["Year"].max())
months = range(1,12)
days = range (1, 31)

#possible locations, 20.0 away from the actural tropical storm location
lats =  np.arange(data["LAT"].min(), data["LAT"].max(), 20.0)
longs = np.arange(data["LONG"].min(), data["LONG"].max(), 20.0)

new_rows = pd.DataFrame(product(years, months, days, lats, longs), columns= ["Year", "MONTH", "DAY", "LAT", "LONG"])

#only keeping the rows with valid dates from the new_rows
new_rows = new_rows[new_rows.apply(valid_date, axis=1)]

#Dropping the dates where a storm actually occured
storm_info = data[["Year", "MONTH", "DAY", "LAT", "LONG"]].drop_duplicates()

#comparing the 2 datasets to find the date that is in the new_rows and not in the cleaned tropical storm dataset
merge_data = pd.merge(new_rows, storm_info, how = "left", on= ["Year", "MONTH", "DAY", "LAT", "LONG"], indicator= "merge")
no_storms = merge_data[merge_data["merge"]== "left_only"].drop(columns=["merge"])

#Adding in initilizing values to the no_storm rows
no_storms["storm_occured"] = 0
no_storms["WIND_KTS"] = 0
no_storms["CAT"] = "NA"
no_storms["Storm Intensity"] = 0.0
no_storms["Storm Intensity Label"] = 0

#Using the avg golbal temps and CO2 levels for the no_storms
for column in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]:
    if column in data.columns:
        no_storms[column] = data[column].mean()

#joing the no_storms to the dataset with the storms
join_data = pd.concat([data, no_storms], ignore_index= True)

#Returning all the rows in a random order
shuffle_data = join_data.sample(frac=1).reset_index(drop=True)

#Updating the csv with the no_storms
shuffle_data.to_csv("completed_dataset_for_IS_project_25.csv", index= False)
'''

'\ndata["storm_occured"] = 1\n\n#possible dates\nyears = range(data["Year"].min(), data["Year"].max())\nmonths = range(1,12)\ndays = range (1, 31)\n\n#possible locations, 20.0 away from the actural tropical storm location\nlats =  np.arange(data["LAT"].min(), data["LAT"].max(), 20.0)\nlongs = np.arange(data["LONG"].min(), data["LONG"].max(), 20.0)\n\nnew_rows = pd.DataFrame(product(years, months, days, lats, longs), columns= ["Year", "MONTH", "DAY", "LAT", "LONG"])\n\n#only keeping the rows with valid dates from the new_rows\nnew_rows = new_rows[new_rows.apply(valid_date, axis=1)]\n\n#Dropping the dates where a storm actually occured\nstorm_info = data[["Year", "MONTH", "DAY", "LAT", "LONG"]].drop_duplicates()\n\n#comparing the 2 datasets to find the date that is in the new_rows and not in the cleaned tropical storm dataset\nmerge_data = pd.merge(new_rows, storm_info, how = "left", on= ["Year", "MONTH", "DAY", "LAT", "LONG"], indicator= "merge")\nno_storms = merge_data[merge_data["merg

Using the data in the dataset where the Storm Intensity Label is greater than 1 and less than 7 to represent tropical storm records the the lables less than and equal to 1 to represent the data for no storm occured 

In [83]:
#data = data[data.apply(valid_date, axis=1)]

data["storm_occured"] = data["Storm Intensity Label"].apply(lambda val: 1 if 1 < val <= 7 else 0)

Checking if the no storm data was added into the dataset

In [84]:
#data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

no_storm = data[data["storm_occured"]== 0]
yes_storm = data[data["storm_occured"]== 1]

print(no_storm.shape, yes_storm.shape)

(14017, 27) (38639, 27)


The data is unbalanced leaning to the yes storm data which would cause our model to have a bias to yes storm, will have to balance the data for a better prediction 

In [85]:
yes_storm = yes_storm.head(no_storm.shape[0])
print(no_storm.shape, yes_storm.shape)

(14017, 27) (14017, 27)


Joining and shuffling the remaining data

In [86]:
data = pd.concat([no_storm, yes_storm])
data = shuffle(data)
data.head(5)

,Year,MONTH,DAY,LAT,LONG,WIND_KTS,PRESSURE,CAT,Shape_Leng,Country,...,Jul,Aug,Sep,Oct,Nov,Dec,Storm Intensity,Storm Intensity Label,Wind Speed Squared,storm_occured
33800,1985,8,3,16.6,-137.8,25,0,TD,0.761577,United States,...,0.04,0.17,0.13,0.12,0.05,0.14,19.039425,1,625,0
14463,1947,8,1,24.1,-96.2,35,0,TS,1.029563,Mexico,...,-0.04,-0.07,-0.12,0.07,0.03,-0.13,36.034705,2,1225,1
9512,1924,8,18,15.2,-62.9,30,0,TD,1.000000,Guadeloupe,...,-0.29,-0.36,-0.32,-0.35,-0.21,-0.43,30.000000,1,900,0
29751,1978,8,5,25.1,-91.0,20,1012,TD,0.854400,Mexico,...,0.04,-0.13,0.06,0.03,0.14,0.08,17.088000,1,400,0
33485,1984,9,1,16.9,-59.1,30,1012,TD,0.707107,Guadeloupe,...,0.19,0.19,0.21,0.14,0.07,-0.04,21.213210,1,900,0


Training the model

In [87]:
features = ["Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "storm_occured"

X= data[features]
y = data[target]

#Spliting the merged dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5, stratify=y) #for balance split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5, stratify=y_temp)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")


Total size: 28034
Train size: 19623
Validation size: 4205
Test size: 4206


Training the Model

In [88]:
#dropping rows with missing values
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_val = X_val.dropna()
y_val = y_val.loc[X_val.index]

print(y_train.value_counts())

model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=5,max_features= 5 )
model.fit(X_train, y_train)

storm_occured
0    9812
1    9811
Name: count, dtype: int64


GradientBoostingClassifier(max_features=5, random_state=5)

Evaluation using the test and validation data

Classification → Accuracy, Precision
Efficiency → Latency


In [89]:
#Test
start_time_test = time.time()
y_test_prediction = model.predict(X_test)
latency_test = time.time() - start_time_test

accuracy_test = accuracy_score(y_test, y_test_prediction)
precision_test = precision_score(y_test, y_test_prediction)

print("Evaluation Test")
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Latency: {latency_test}")

Evaluation Test
Accuracy: 0.9060865430337612
Precision: 0.8435237329042639
Latency: 0.015399456024169922


In [90]:
#Validation
start_time_val = time.time()
y_val_prediction = model.predict(X_val)
latency_val = time.time() - start_time_val

accuracy_val = accuracy_score(y_val, y_val_prediction)
precision_val = precision_score(y_val, y_val_prediction)

print("Evaluation Validation")
print(f"Accuracy: {accuracy_val}")
print(f"Precision: {precision_val}")
print(f"Latency: {latency_val}")

Evaluation Validation
Accuracy: 0.9084423305588585
Precision: 0.8460918614020951
Latency: 0.01801919937133789


For the user to add in their variables to predict if a storm would occur or not and if a storm does occur it would lead the user into the other model to predict the intensity

In [91]:
def predict_storm(model, year, month, day, co2, temps_dict):
    input = {
        "Year": [year],
        "MONTH": [month],
        "DAY": [day],
        "CO2 emission (Tons)": [co2],
    }

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
    
    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, 0)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    return prediction #0 or 1 for the model in part 2

In [92]:
#example for predicting a possible storm
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, year=1880, month=8, day=14, co2= 2667491000.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")

A storm was predicted


In [93]:
'''
#average global temps and CO2 levels
avg_column = ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]
avg = {col: data[col].mean() for col in avg_column if col in data.columns}
 
possible_temps= {month: avg[month] for month in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]}
avg_co2 = avg["CO2 emission (Tons)"]
'''

'\n#average global temps and CO2 levels\navg_column = ["Jan", "Feb", "Mar",\t"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]\navg = {col: data[col].mean() for col in avg_column if col in data.columns}\n \npossible_temps= {month: avg[month] for month in ["Jan", "Feb", "Mar",\t"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]}\navg_co2 = avg["CO2 emission (Tons)"]\n'

For no storm that input data would have lower CO2 levels and and lower global temps

In [94]:
#example for predicting a no storm
possible_temps= {"Jan": -1.5, "Feb": -1.3, "Mar": -1.1, "Apr": -1.2, "May": -1.0, "Jun": -1.4, "Jul": -1.3, "Aug": -1.5, "Sep": -1.2, "Oct": -1.3, "Nov": -1.4, "Dec": -1.6}
will_storm_occur = predict_storm(model=model, year=1890, month=1, day=5, co2= 15000000.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")


No storm was predicted


In [95]:
print("Classification Report for Model 1 - Gradient Boosting Classifier")

print("Test Report")
print(classification_report(y_test, y_test_prediction))

print("Validation Report")
print(classification_report(y_val, y_val_prediction))

Classification Report for Model 1 - Gradient Boosting Classifier
Test Report
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      2103
           1       0.84      1.00      0.91      2103

    accuracy                           0.91      4206
   macro avg       0.92      0.91      0.91      4206
weighted avg       0.92      0.91      0.91      4206

Validation Report
              precision    recall  f1-score   support

           0       1.00      0.82      0.90      2102
           1       0.85      1.00      0.92      2103

    accuracy                           0.91      4205
   macro avg       0.92      0.91      0.91      4205
weighted avg       0.92      0.91      0.91      4205



Random Forest Regressor

Part 2: Predicting the intensity of the storm using Random Forest Regressor

Objective: 
    min Z2 error in predicting the intensity of the tropical storm
Constraints:
	    Wind_kts, pressure, category >= 0 
        stormed_occured =1
	    Storm Intensity Label >= 1 && Storm Intensity Label <=5

In [96]:
#Only accessing the rows were a storm occurred to train the models on the intensity of the tropical storms

storms = data[(data["storm_occured"] ==1) & (data["Storm Intensity Label"] >=1) &(data["Storm Intensity Label"] <=5)].copy()

In [97]:
#Converting the CAT values to int for machine learning
le = LabelEncoder()
storms["CAT_int"] = le.fit_transform(storms["CAT"].astype(str))

Training the model

In [98]:
#features_2 = storms[["CAT_int", "Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"]]
#features_2 = storms[["Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"]]
features_2 = storms[["Year", "MONTH", "DAY", "CO2 emission (Tons)"]]
target_2 = storms["Storm Intensity Label"]

#3. removing duplicates
no_duplicates = pd.concat([features_2,target_2],axis=1)
#no_duplicates= no_duplicates.drop_duplicates(subset=["CAT_int", "Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"])
#no_duplicates= no_duplicates.drop_duplicates(subset=["Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"])
no_duplicates= no_duplicates.drop_duplicates(subset=["Year", "MONTH", "DAY", "CO2 emission (Tons)"])

#X= no_duplicates[["CAT_int", "Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"]]
#X= no_duplicates[["Year", "MONTH", "DAY", "CO2 emission (Tons)", "WIND_KTS"]]
X= no_duplicates[["Year", "MONTH", "DAY", "CO2 emission (Tons)"]]
y= no_duplicates["Storm Intensity Label"]

#4. Still getting 0.0 for the MSE and MAE, the CAT_int and Storm Intensity Label could be closely related
'''
sns.boxplot(x=X["CAT_int"], y=y)
plt.title("Storm Intensity Lable VS Cat_int")
plt.show()
'''
#The results showed that they were directly mapped to each other solution, remove CAT_int from the features of this model

#2. Currently the X_train and Z_test has 1861 rows in common, shuffing and resetting to help fix the issue 
X, y = shuffle(X,y, random_state=5)
X.reset_index(drop=True, inplace=True)
y.reset_index(drop=True, inplace=True)

#Spliting the dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

#1. Currently getting 0.0 for both the MSE and the MAE, checking for overlaps
both_sets= pd.merge(X_train, X_test, how= "inner")
print(f"Overlaps in sets: {len(both_sets)} rows")

#model_2 = LogisticRegression(max_iter=1000, random_state=5)
model_2 = RandomForestRegressor(n_estimators=100, random_state=5)
#model_2 = RandomForestClassifier(n_estimators=100, random_state=5)
#model_2 = GradientBoostingRegressor(n_estimators=100, random_state=5)
model_2.fit(X_train, y_train)

Total size: 4495
Train size: 3146
Validation size: 674
Test size: 675
Overlaps in sets: 0 rows


RandomForestRegressor(random_state=5)

Evaluation for the test and validation

Regression → MSE, MAE, R-squared 
Efficiency → Latency

In [99]:
#test
start_test_time_model_2 = time.time()
y_test_prediction_model_2 = model_2.predict(X_test)
latency_test_model_2 = time.time() - start_test_time_model_2

mse_test_model_2 = mean_squared_error(y_test, y_test_prediction_model_2)
mae_test_model_2 = mean_absolute_error(y_test, y_test_prediction_model_2)
r2_test_model_2 = r2_score(y_test, y_test_prediction_model_2)

print("Model 2 Test Evaluation")
print(f"Mean Squared Error {mse_test_model_2}")
print(f"Mean Absolute Error {mae_test_model_2}")
print(f"R2 score {r2_test_model_2}")
print(f"Latency {latency_test_model_2}")

Model 2 Test Evaluation
Mean Squared Error 0.4655340740740741
Mean Absolute Error 0.4965481481481482
R2 score 0.4545308224638426
Latency 0.031168460845947266


In [100]:
#validation
start_val_time_model_2 = time.time()
y_val_prediction_model_2 = model_2.predict(X_val)
latency_val_model_2 = time.time() - start_val_time_model_2

mse_val_model_2 = mean_squared_error(y_val, y_val_prediction_model_2)
mae_val_model_2 = mean_absolute_error(y_val, y_val_prediction_model_2)
r2_val_model_2 = r2_score(y_val, y_val_prediction_model_2)

print("Model 2 Val Evaluation")
print(f"Mean Squared Error {mse_val_model_2}")
print(f"Mean Absolute Error {mae_val_model_2}")
print(f"R2 score {r2_val_model_2}")
print(f"Latency {latency_val_model_2}")

Model 2 Val Evaluation
Mean Squared Error 0.5457446587537093
Mean Absolute Error 0.5412908011869437
R2 score 0.3917717498184531
Latency 0.029372692108154297


For the user to add in their variables to predict the insensity of a storm 

In [101]:
#def predict_intensity(model, wind_kts, cat, co2, year, month, day):
#def predict_intensity(model, wind_kts, co2, year, month, day):
def predict_intensity(model, co2, year, month, day):
    #cat_int = le.transform([cat])[0]
    input = pd.DataFrame([{
    #"CAT_int": cat_int,
    "Year": year,
    "MONTH":month,
    "DAY": day,
    "CO2 emission (Tons)": co2,
    #"WIND_KTS": wind_kts
    }])

    return model.predict(input)[0]

In [102]:
#example for predicting a possible storm
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, year=1880, month=8, day=14, co2= 2667491000.0, temps_dict = possible_temps)

if will_storm_occur== 1:
    print("A storm was predicted, now predicting it's intensity...")
    #intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    #intensity = predict_intensity(model = model_2, wind_kts=70, co2=412.5, year=2024, month=8, day=14)
    intensity = predict_intensity(model = model_2, co2=412.5, year=2024, month=8, day=14)

    #Before rounding
    #print(f"Predicted storm intensity: {intensity}")
    #print("Based on MyNASEData Hurricane Dynamics")
    
    #Some times I get the intensity as a decimal, using this to round to the nearest whole number
    intensity = round(intensity)
    
    if intensity == -2:
        value = "Warning flag"
    elif intensity == -1:
        value = "Extra-tropical (non-standard storm)"
    elif intensity == 0:
        value = "Low-pressure system"
    elif intensity == 1:
        value = "Tropical Depression"
    elif intensity == 2:
        value = "Tropical Storm"
    elif intensity == 3:
        value = "Hurricane Category 1"
    elif intensity == 4:
        value = "Hurricane Category 2"
    elif intensity == 5:
        value = "Hurricane Category 3"
    elif intensity == 6:
        value = "Hurricane Category 4"
    elif intensity == 7:
        value = "Hurricane Category 5"
    else:
        value = "Unknown"

    print(f"Predicted storm intensity: {value} with an intensity of {intensity}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

A storm was predicted, now predicting it's intensity...
Predicted storm intensity: Tropical Storm with an intensity of 2


In [103]:
'''
#average global temps and CO2 levels
avg_column = ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]
avg = {col: data[col].mean() for col in avg_column if col in data.columns}
 
possible_temps= {month: avg[month] for month in ["Jan", "Feb", "Mar",	"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]}
avg_co2 = avg["CO2 emission (Tons)"]
'''

'\n#average global temps and CO2 levels\navg_column = ["Jan", "Feb", "Mar",\t"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec", "CO2 emission (Tons)"]\navg = {col: data[col].mean() for col in avg_column if col in data.columns}\n \npossible_temps= {month: avg[month] for month in ["Jan", "Feb", "Mar",\t"Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]}\navg_co2 = avg["CO2 emission (Tons)"]\n'

In [104]:
#example for predicting a no storm
possible_temps= {"Jan": -1.5, "Feb": -1.3, "Mar": -1.1, "Apr": -1.2, "May": -1.0, "Jun": -1.4, "Jul": -1.3, "Aug": -1.5, "Sep": -1.2, "Oct": -1.3, "Nov": -1.4, "Dec": -1.6}
will_storm_occur = predict_storm(model=model, year=1890, month=1, day=5, co2= 15000000.0, temps_dict = possible_temps)

if will_storm_occur== 1:
    print("A storm was predicted, now predicting it's intensity...")
    #intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    #intensity = predict_intensity(model = model_2, wind_kts=70, co2=412.5, year=2024, month=8, day=14)
    intensity = predict_intensity(model = model_2, co2=412.5, year=2024, month=8, day=14)

    #print(f"Predicted storm intensity: {intensity}")
    #print("Based on MyNASEData Hurricane Dynamics")
    
    intensity = round(intensity)
    
    if intensity == -2:
        value = "Warning flag"
    elif intensity == -1:
        value = "Extra-tropical (non-standard storm)"
    elif intensity == 0:
        value = "Low-pressure system"
    elif intensity == 1:
        value = "Tropical Depression"
    elif intensity == 2:
        value = "Tropical Storm"
    elif intensity == 3:
        value = "Hurricane Category 1"
    elif intensity == 4:
        value = "Hurricane Category 2"
    elif intensity == 5:
        value = "Hurricane Category 3"
    elif intensity == 6:
        value = "Hurricane Category 4"
    elif intensity == 7:
        value = "Hurricane Category 5"
    else:
        value = "Unknown"

    print(f"Predicted storm intensity: {value}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

No storm was predicted, skipping the intensity prediction
